# Session 2: Multiple Views

A single spatial view can display layered images, segmentations, spots, and points.
However, Vitessce is designed to display **multiple views simultaneously**. Each view may show different data, or a different visual representation of the same underlying data.

In this notebook we:

1. Introduce additional Vitessce view types
2. Learn how to arrange views with the layout function
3. Build a multi-view dashboard for a single-cell dataset
4. Combine spatial and non-spatial views for a spatial dataset
5. Learn how these topics relate to EasyVitessce

In [ ]:
!pip install "vitessce[all]==3.9.2" "scanpy" "easy-vitessce==0.0.12" "spatialdata==0.7.3" "spatialdata-plot==0.4.0"

## View types overview

Vitessce provides many view types. The most commonly used ones for single-cell and spatial data are:

| View type constant | String name | What it shows |
|---|---|---|
| `vt.SCATTERPLOT` | `"scatterplot"` | 2D embedding (UMAP, PCA, t-SNE) |
| `vt.HEATMAP` | `"heatmap"` | Cell-by-gene expression heatmap |
| `vt.OBS_SETS` | `"obsSets"` | View and select cell types or clusters |
| `vt.FEATURE_LIST` | `"featureList"` | Searchable gene list |
| `vt.OBS_SET_FEATURE_VALUE_DISTRIBUTION` | `"obsSetFeatureValueDistribution"` | Violin plot per observation set (i.e., cell type or cluster) |
| `vt.DOT_PLOT` | `"dotPlot"` | Dot plot (mean expression and percent-expressing for multiple cell types and genes) |
| `"spatialBeta"` | `"spatialBeta"` | Spatial and imaging view |
| `"layerControllerBeta"` | `"layerControllerBeta"` | Spatial layer and channel controls |

The `vt` view type constants can be imported with `from vitessce import ViewType as vt`.

## Layout operators

The `VitessceConfig.layout` function supports using Python's `/` and `|` operators to compose views into a 12-by-12 grid:

| Expression | Result |
|---|---|
| `view1 \| view2` | `view1` and `view2` side-by-side (horizontal split) |
| `view1 / view2` | `view1` above `view2` below (vertical split) |
| `view1 \| (view2 / view3)` | `view1` on the top half; `view2` bottom-left; `view3` on the bottom-right |
| `(view1 \| view2) / view3` | `view1` and `view2` side-by-side on top; `view3` spans the full width below |

```python
# Pass the layout expression to config.layout()
config.layout(umap | (obs_sets / feature_list))
```

The `/` operator here is not division -- Vitessce overrides it on view objects to mean "above / below".

### `hconcat` and `vconcat`

The slash and pipe operators are concise, but due to being operators, they short-circuit so `A | B | C` will effectively result in `(A | B) | C`. To avoid this issue, you can use the `hconcat` ("horizontally concatenate") and `vconcat` ("vertically concatenate") functions.

```python
from vitessce import hconcat, vconcat

config.layout(hconcat(umap, vconcat(obs_sets, feature_list))) # equivalent to the above
```

Further, `hconcat` and `vconcat` support providing a `split` parameter to control the "weight" of each view when being concatenated together to fit into the grid space.

For example, `hconcat(view1, view2, view3, split=[1, 2, 1])` will result in `view2` being twice as wide as `view1` and `view3`.

## Example 1: PBMC single-cell data

We will use the [PBMC 68k reduced](https://scanpy.readthedocs.io/en/stable/generated/scanpy.datasets.pbmc68k_reduced.html) dataset that ships with Scanpy. It contains ~500 peripheral blood mononuclear cells (PBMCs) with precomputed UMAP and PCA embeddings and Leiden cluster labels.

### Step 1 — Load and prepare the data

The `AnnDataWrapper` expects an [AnnData Zarr store](https://anndata.readthedocs.io/en/latest/fileformat-prose.html) on disk.

In [ ]:
import os
from os.path import join, isdir
import scanpy as sc

from vitessce.data_utils import VAR_CHUNK_SIZE

# Load Scanpy's built-in reduced PBMC dataset.
adata = sc.datasets.pbmc68k_reduced()
sc.tl.tsne(adata)

print(adata)
print("\nAvailable embeddings:", list(adata.obsm.keys()))
print("Available obs columns:", list(adata.obs.columns))

In [ ]:
zarr_path = join("data", "pbmc68k.zarr")

if not isdir(zarr_path):
    os.makedirs("data", exist_ok=True)
    adata.write_zarr(zarr_path, chunks=[adata.shape[0], VAR_CHUNK_SIZE])

print("Zarr store ready:", zarr_path)

### Step 2 — Build the configuration

`AnnDataWrapper` tells Vitessce how to read each part of the AnnData object:

- `obs_set_paths` / `obs_set_names` — the cell cluster columns (shown in the OBS_SETS view)
- `obs_embedding_paths` / `obs_embedding_names` — the dimensionality-reduction arrays (shown as scatterplots)
- `obs_feature_matrix_path` — the gene expression matrix (shown in the heatmap and used for gene coloring)

In [ ]:
from vitessce import VitessceConfig, ViewType as vt, AnnDataWrapper

vc = VitessceConfig(schema_version="1.0.18", name="PBMC 68k dataset with Multiple Views")

dataset = vc.add_dataset(name="PBMC 68k").add_object(
    AnnDataWrapper(
        adata_store=zarr_path,
        obs_set_paths=["obs/bulk_labels", "obs/louvain"],
        obs_set_names=["Cell Type", "Leiden Cluster"],
        obs_embedding_paths=["obsm/X_umap", "obsm/X_pca"],
        obs_embedding_names=["UMAP", "PCA"],
        obs_feature_matrix_path="X",
    )
)

# --- Define views ---
umap      = vc.add_view(vt.SCATTERPLOT, dataset=dataset)
pca       = vc.add_view(vt.SCATTERPLOT, dataset=dataset)
obs_sets  = vc.add_view(vt.OBS_SETS, dataset=dataset)
genes     = vc.add_view(vt.FEATURE_LIST, dataset=dataset)
heatmap   = vc.add_view(vt.HEATMAP, dataset=dataset)

vc.link_views_by_dict([umap], { "embeddingType": "UMAP" })
vc.link_views_by_dict([pca], { "embeddingType": "PCA" })

# --- Define the layout ---
# UMAP on the top-left, PCA top-right, cell sets and gene list bottom-right, heatmap on the bottom-left
vc.layout((umap | pca) / (heatmap | (obs_sets / genes)))
vc.widget()

### Exercises

👉 **Modify the layout expression above** so that the heatmap takes up the full width or height of the grid.

👉 **Your own layout:** Rearrange the five views (`umap`, `pca`, `obs_sets`, `genes`, `heatmap`) any way you like using `|` and `/` or `hconcat` and `vconcat`.

## Example 2: Combining spatial and non-spatial views

Spatial transcriptomics datasets have both a tissue image (the spatial component) and a gene expression matrix (the non-spatial component). Vitessce can show both in the same dashboard.

We will re-use the Visium SpatialData object from Session 1. If you haven't run Session 1 yet, run the download cell from that notebook first to create `data/visium.spatialdata.zarr`.

The `SpatialDataWrapper` supports all the same AnnData-style fields as `AnnDataWrapper`, so we can add a UMAP scatterplot alongside the spatial view — provided the SpatialData table contains an embedding.

In [ ]:
import os
from os.path import join, isfile, isdir
from urllib.request import urlretrieve
import zipfile

data_dir = "data"
sdata_filepath = join(data_dir, "visium.spatialdata.zarr")

# Download if not already present (same code as Session 1)
if not isdir(sdata_filepath):
    zip_filepath = join(data_dir, "visium.spatialdata.zarr.zip")
    if not isfile(zip_filepath):
        os.makedirs(data_dir, exist_ok=True)
        urlretrieve(
            "https://data-2.vitessce.io/sdata-datasets/visium.spatialdata.zarr.zip",
            zip_filepath,
        )
    with zipfile.ZipFile(zip_filepath, "r") as zip_ref:
        zip_ref.extractall(data_dir)
        os.rename(join(data_dir, "data.zarr"), sdata_filepath)

print("Visium dataset ready.")

In [ ]:
from vitessce import (
    VitessceConfig,
    ViewType as vt,
    SpatialDataWrapper,
    vconcat, hconcat,
)

vc2 = VitessceConfig(schema_version="1.0.18", name="Visium example")

wrapper = SpatialDataWrapper(
    sdata_store=sdata_filepath,
    image_path="images/ST8059050_hires_image",
    obs_spots_path="shapes/ST8059050",
    table_path="tables/table",
    obs_feature_matrix_path="tables/table/X",
    coordinate_system="ST8059050",
    coordination_values={"obsType": "spot"},
)
dataset2 = vc2.add_dataset(name="Visium").add_object(wrapper)

# Spatial views
spatial   = vc2.add_view("spatialBeta", dataset=dataset2)
lc        = vc2.add_view("layerControllerBeta", dataset=dataset2)

# Non-spatial views
obs_sets2 = vc2.add_view(vt.OBS_SETS, dataset=dataset2)
genes2    = vc2.add_view(vt.FEATURE_LIST, dataset=dataset2)
heatmap2  = vc2.add_view(vt.HEATMAP, dataset=dataset2)

# Link obsType so all views refer to "spot"
vc2.link_views([spatial, lc, obs_sets2, genes2, heatmap2], ["obsType"], [wrapper.obs_type_label])

vc2.layout(spatial | (vconcat(lc, obs_sets2, genes2) / heatmap2))
vc2.widget()

## The EasyVitessce approach

Similar PBMC visualizations can be created with far less code using [EasyVitessce](https://vitessce.github.io/easy_vitessce/easy_vitessce.html)'s support for the Scanpy plotting APIs. When `easy_vitessce` is imported, Scanpy functions like `sc.pl.embedding()` and `sc.pl.dotplot()` automatically return interactive Vitessce widgets instead of static matplotlib figures.

In [ ]:
import easy_vitessce as ev  # intercepts scanpy plotting calls
import scanpy as sc

# This line is required when the notebook kernel is running on a different machine (e.g., Google Colab, Docker container, or HPC cluster)
ev.config.set({ 'data.wrapper_param_suffix': '_store' })

adata = sc.datasets.pbmc68k_reduced()

# Interactive UMAP colored by cell type
sc.pl.embedding(adata, basis="umap", color="bulk_labels")

In [ ]:
# Show multiple genes side-by-side in separate panels
sc.pl.embedding(adata, basis="umap", color=["CD79A", "CD53", "CLIC1", "ANXA1"], ncols=2)

### Exercise 2

👉 **Try the following modifications:**

1. **Change the coloring** in `sc.pl.embedding()` from `"bulk_labels"` to a gene name such as `"CST3"` or `"LYZ"`.

2. **Add a dot plot** by running the cell below. A dot plot shows mean expression and percentage of expressing cells for a selected set of genes grouped by cell type.

```python
sc.pl.dotplot(
    adata,
    var_names=["CD79A", "CD79B", "CST3", "LYZ", "PSAP"],
    groupby="bulk_labels",
)
```

3. **Add a violin plot** showing the expression distribution of one gene across cell types:

```python
sc.pl.violin(adata, keys="LYZ", groupby="bulk_labels")
```


In [ ]:
sc.pl.embedding(adata, basis="umap", color="???")  # Change the color